In [6]:
from qiskit.circuit.library import CXGate, RXXGate, SwapGate, iSwapGate
import numpy as np
from gulps.invariants import (
    mono_coordinates_to_CAN,
    mono_coordinates_to_makhlin,
    positive_canonical_to_monodromy_coordinate,
    recover_local_equivalence,
)
from gulps.lm_numerics import _SegmentNumericSynthesis

In [7]:
# Example usage
# Define a gate list and invariant list for testing

gate_list = [CXGate(), CXGate(), CXGate()]
invariant_list = np.pi / 2 * np.array([(0.5, 0.5, 0.0), (0.5, 0.5, 0), (0.5, 0.5, 0.5)])
invariant_list = [
    positive_canonical_to_monodromy_coordinate(*c) for c in invariant_list
]

# Synthesize segments
segment_solutions = _SegmentNumericSynthesis._synthesize_segments(
    gate_list, invariant_list
)
print("Segment solutions:", segment_solutions)

Segment solutions: [array([ 0.70127154,  4.58352926, -0.70072199,  3.05288158,  2.49471306,
        2.04311779]), array([ 9.91573203,  9.07515998, -9.91573279,  0.03703351,  3.97905326,
       -4.86252828])]


In [8]:
ret = _SegmentNumericSynthesis._stitch_segments(
    gate_list, invariant_list, segment_solutions
)
from weylchamber import c1c2c3
from qiskit.quantum_info import Operator

print(c1c2c3(Operator(ret).data))
ret.draw()

(0.5, 0.5, 0.49999998)


global phase: 5π/4
     ┌─────────┐┌─────────┐       ┌──────────────────────────┐      ┌─────────┐»
q_0: ┤ Unitary ├┤ Unitary ├──■────┤ Rv(3.0529,2.4947,2.0431) ├───■──┤ Unitary ├»
     ├─────────┤├─────────┤┌─┴─┐┌─┴──────────────────────────┴┐┌─┴─┐├─────────┤»
q_1: ┤ Unitary ├┤ Unitary ├┤ X ├┤ Rv(0.70127,4.5835,-0.70072) ├┤ X ├┤ Unitary ├»
     └─────────┘└─────────┘└───┘└─────────────────────────────┘└───┘└─────────┘»
«     ┌─────────────────────────────┐     ┌─────────┐
«q_0: ┤ Rv(0.037034,3.9791,-4.8625) ├──■──┤ Unitary ├
«     └┬───────────────────────────┬┘┌─┴─┐├─────────┤
«q_1: ─┤ Rv(9.9157,9.0752,-9.9157) ├─┤ X ├┤ Unitary ├
«      └───────────────────────────┘ └───┘└─────────┘

In [9]:
from qutip import Qobj
from qiskit.quantum_info import Operator, average_gate_fidelity

lm_synthesizer = _SegmentNumericSynthesis()
decomp = lm_synthesizer(gate_list, invariant_list, target_unitary=SwapGate())
decomp_u = Operator(decomp)
decomp.draw()
print(average_gate_fidelity(SwapGate(), decomp_u))
Qobj(decomp_u.data)

0.9999999999999994


Quantum object: dims=[[4], [4]], shape=(4, 4), type='oper', dtype=Dense, isherm=False
Qobj data =
[[ 1.00000000e+00-2.94231712e-08j -1.67418252e-16+3.33066907e-16j
   9.27887926e-17-1.61233642e-17j -7.62519697e-10-6.56204044e-16j]
 [ 4.19981953e-16-2.74578004e-16j -7.62520820e-10+6.34374881e-16j
   1.00000000e+00+2.94231710e-08j -1.93119723e-17+2.77555756e-16j]
 [-2.89870075e-16+2.22044605e-16j  1.00000000e+00+2.94231715e-08j
   7.62520596e-10+9.22018453e-16j -9.60774769e-17+1.04188583e-16j]
 [ 7.62519729e-10-8.21832762e-16j  5.11901573e-16-2.15040128e-16j
  -1.00411046e-16+2.77555756e-16j  1.00000000e+00-2.94231717e-08j]]